In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression

# Paths
raw_data_path = "/Users/khoale/Downloads/Alzheimer_Code/csvs/adni_mri_ucsf_merged.csv"
demographics_path = "/Users/khoale/Downloads/Alzheimer_Code/TABLE_DATA_ADNI/MRI_T1_3D_ADNI1_2_BL_PRE_4_24_2026.csv"
output_path = "/Users/khoale/Downloads/Alzheimer_Code/adni_mri_sustain_prepared_asymmetric_as.csv"

# Load the raw dataset
df = pd.read_csv(raw_data_path)
print(f"Loaded raw data: {df.shape[0]} subjects, {df.shape[1]} columns")

# Load demographics and subset columns
dem_df = pd.read_csv(demographics_path)[['Image Data ID', 'Age', 'Sex']]

# Merge using Image ID as the key
df = df.merge(dem_df, left_on='MRI_ImageID', right_on='Image Data ID', how='left')
print(f"Merged demographics successfully by Image ID.")
print(f"  Missing values after merge - Age: {df['Age'].isna().sum()}, Sex: {df['Sex'].isna().sum()}")

Loaded raw data: 577 subjects, 355 columns
Merged demographics successfully by Image ID.
  Missing values after merge - Age: 0, Sex: 0


In [2]:
# Map columns belonging to each of our 24 asymmetric biomarkers
biomarker_mapping = {
    # --- LEFT HEMISPHERE ---
    "L_Frontal": ["ST15CV", "ST25CV", "ST36CV", "ST39CV", "ST43CV", "ST45CV", "ST46CV", "ST47CV", "ST51CV", "ST55CV", "ST56CV"],
    "L_Temporal": ["ST13CV", "ST24CV", "ST26CV", "ST32CV", "ST40CV", "ST44CV", "ST58CV", "ST60CV", "ST62CV"],
    "L_Parietal": ["ST31CV", "ST49CV", "ST52CV", "ST57CV", "ST59CV"],
    "L_Occipital": ["ST23CV", "ST35CV", "ST38CV", "ST48CV"],
    "L_Cingulate": ["ST14CV", "ST34CV", "ST50CV", "ST54CV"],
    "L_Insula": ["ST129CV"],
    "L_Hippocampus": ["ST29SV"],
    "L_Amygdala": ["ST12SV"],
    "L_Caudate": ["ST16SV"],
    "L_Pallidum": ["ST42SV"],
    "L_Putamen": ["ST53SV"],
    "L_Accumbens": ["ST11SV"],

    # --- RIGHT HEMISPHERE ---
    "R_Frontal": ["ST74CV", "ST84CV", "ST95CV", "ST98CV", "ST102CV", "ST104CV", "ST105CV", "ST106CV", "ST110CV", "ST114CV", "ST115CV"],
    "R_Temporal": ["ST72CV", "ST83CV", "ST85CV", "ST91CV", "ST99CV", "ST103CV", "ST117CV", "ST119CV", "ST121CV"],
    "R_Parietal": ["ST90CV", "ST108CV", "ST111CV", "ST116CV", "ST118CV"],
    "R_Occipital": ["ST82CV", "ST94CV", "ST97CV", "ST107CV"],
    "R_Cingulate": ["ST73CV", "ST93CV", "ST109CV", "ST113CV"],
    "R_Insula": ["ST130CV"],
    "R_Hippocampus": ["ST88SV"],
    "R_Amygdala": ["ST71SV"],
    "R_Caudate": ["ST75SV"],
    "R_Pallidum": ["ST101SV"],
    "R_Putamen": ["ST112SV"],
    "R_Accumbens": ["ST70SV"]
}

In [3]:
# 1. Force all ROI columns and ICV (ST10CV) to numeric values
all_roi_cols = [col for cols in biomarker_mapping.values() for col in cols]
cols_to_convert = all_roi_cols + ["ST10CV"]

for col in cols_to_convert:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Drop rows where Intracranial Volume (ICV) is missing
df = df.dropna(subset=["ST10CV"]).copy()

# 3. Sum sub-regions for Left and Right separately, keeping covariates
aggregated_df = pd.DataFrame()
aggregated_df["PTID"] = df["PTID"]
aggregated_df["Label"] = df["Label"]
aggregated_df["ICV"] = df["ST10CV"]
aggregated_df["AGE"] = df["Age"]  # Casing matches loaded 'Age' column
# Map Sex: M/Male -> 1, F/Female -> 0
aggregated_df["Sex_Code"] = df["Sex"].map({'Male': 1, 'Female': 0, 'M': 1, 'F': 0})

for region, cols in biomarker_mapping.items():
    aggregated_df[region] = df[cols].sum(axis=1, min_count=1)

print(f"Aggregated asymmetric data shape: {aggregated_df.shape}")
aggregated_df.head()

Aggregated asymmetric data shape: (577, 29)


,PTID,Label,ICV,AGE,Sex_Code,L_Frontal,L_Temporal,L_Parietal,L_Occipital,L_Cingulate,...,R_Parietal,R_Occipital,R_Cingulate,R_Insula,R_Hippocampus,R_Amygdala,R_Caudate,R_Pallidum,R_Putamen,R_Accumbens
0,003_S_1057,pMCI,1.352138e+06,61,0,72851.0,50525.0,42733.0,20813.0,9146.0,...,42838.0,18902.0,7182.0,5530.0,3270.5,1159.7,3268.1,1897.3,4703.5,418.7
1,003_S_1059,AD,1.520293e+06,85,0,57614.0,39750.0,39752.0,20244.0,7849.0,...,40367.0,21093.0,6507.0,6222.0,2668.8,877.1,3165.2,1482.6,2885.1,296.9
2,003_S_1257,AD,1.951182e+06,85,1,73453.0,47058.0,43438.0,22424.0,7604.0,...,48298.0,24454.0,6380.0,7278.0,4261.7,1700.3,5461.0,1741.9,3455.8,318.1
3,003_S_4119,CN,1.475697e+06,79,1,83368.0,55766.0,49947.0,21359.0,9471.0,...,53686.0,21678.0,9403.0,6945.0,3812.5,1713.2,3695.0,1801.5,4378.5,428.8
4,003_S_4152,AD,1.383684e+06,61,1,68657.0,39745.0,39933.0,26387.0,9903.0,...,43430.0,28127.0,9055.0,6660.0,3886.2,1419.3,2610.1,2338.7,4187.5,507.1


In [4]:
# Create a copy to store normalized volumes
normalized_df = aggregated_df.copy()

# Define the control (CN) cohort to fit the regression line
cn_controls = aggregated_df[aggregated_df["Label"] == "CN"].copy()

print(f"Fitting ICV + Age + Sex regression using {len(cn_controls)} CN controls...")

regions = list(biomarker_mapping.keys())

for region in regions:
    # Drop NaNs across all regressors
    clean_cn = cn_controls.dropna(subset=[region, "ICV", "AGE", "Sex_Code"])
    
    # Fit regression with ICV, Age, and Sex
    X_cn = clean_cn[["ICV", "AGE", "Sex_Code"]].values
    y_cn = clean_cn[region].values
    
    model = LinearRegression()
    model.fit(X_cn, y_cn)
    
    # Calculate residuals for ALL subjects
    X_all = aggregated_df[["ICV", "AGE", "Sex_Code"]].values
    y_all = aggregated_df[region].values
    
    predicted_all = model.predict(X_all)
    residuals = y_all - predicted_all
    
    # Add back the mean volume of controls to restore volumetric scale
    mean_volume_cn = y_cn.mean()
    normalized_df[region] = residuals + mean_volume_cn

print("Covariates (ICV + Age + Sex) correction complete.")

Fitting ICV + Age + Sex regression using 186 CN controls...
Covariates (ICV + Age + Sex) correction complete.


In [5]:
sustain_ready_df = pd.DataFrame()
sustain_ready_df["PTID"] = normalized_df["PTID"]
sustain_ready_df["Label"] = normalized_df["Label"]

# Fit z-score baseline using the ICV-normalized controls
cn_normalized = normalized_df[normalized_df["Label"] == "CN"]

for region in regions:
    mean_cn = cn_normalized[region].mean()
    std_cn = cn_normalized[region].std() + 1e-9
    
    # Inverted formula for atrophy: Z = (mean_healthy - actual) / std_healthy
    sustain_ready_df[region] = (mean_cn - normalized_df[region]) / std_cn

sustain_ready_df.head()

,PTID,Label,L_Frontal,L_Temporal,L_Parietal,L_Occipital,L_Cingulate,L_Insula,L_Hippocampus,L_Amygdala,...,R_Parietal,R_Occipital,R_Cingulate,R_Insula,R_Hippocampus,R_Amygdala,R_Caudate,R_Pallidum,R_Putamen,R_Accumbens
0,003_S_1057,pMCI,0.933040,0.028028,1.683806,0.724421,0.043507,0.222630,1.468858,2.508597,...,1.819450,1.901451,1.296503,1.193295,2.257800,2.120934,-0.143158,-0.417555,-0.847277,1.335147
1,003_S_1059,AD,2.172895,1.842509,1.531170,-0.353903,0.822974,0.820718,3.221262,2.076538,...,1.533734,-0.134268,1.847897,0.032710,2.195658,2.324408,0.516415,1.339252,2.100341,0.882186
2,003_S_1257,AD,1.427070,2.031658,2.358713,-0.338307,2.446475,2.102724,-2.677484,-1.449546,...,1.462496,-0.319914,3.322411,0.297042,-1.285033,-0.804678,-3.435762,1.483762,2.278140,0.975384
3,003_S_4119,CN,-1.368365,-1.377361,-0.463391,-0.177991,-0.437980,-1.591455,-0.252763,-1.133032,...,-1.036622,0.403620,-0.862761,-0.512600,-0.292998,-0.812762,-0.582795,0.469258,-0.008175,-0.065943
4,003_S_4152,AD,2.273611,3.652299,2.586604,-1.304103,-0.416814,0.080507,2.272619,3.082100,...,2.040005,-1.419258,-0.295749,0.118774,0.837207,1.500941,1.488524,-1.683143,0.834146,0.449187


In [6]:
# 1. Fill missing values (NaNs) with 0 (normal)
sustain_ready_df[regions] = sustain_ready_df[regions].fillna(0)

# 2. Clamp negative z-scores to 0
sustain_ready_df[regions] = sustain_ready_df[regions].clip(lower=0)

print("Preprocessing complete! Preview of final asymmetric z-scores:")
sustain_ready_df[regions].describe()

Preprocessing complete! Preview of final asymmetric z-scores:


,L_Frontal,L_Temporal,L_Parietal,L_Occipital,L_Cingulate,L_Insula,L_Hippocampus,L_Amygdala,L_Caudate,L_Pallidum,...,R_Parietal,R_Occipital,R_Cingulate,R_Insula,R_Hippocampus,R_Amygdala,R_Caudate,R_Pallidum,R_Putamen,R_Accumbens
count,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,...,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000
mean,0.837679,1.279447,0.952323,0.560365,0.718512,0.702597,1.210835,1.111619,0.478360,0.474217,...,0.929945,0.601668,0.633966,0.714194,1.160111,0.959219,0.476734,0.460684,0.610571,0.672402
std,1.138355,1.470238,1.312605,0.755989,1.029179,0.955061,1.239975,1.074616,0.657966,0.690963,...,1.267773,0.798348,1.017602,1.066805,1.245596,1.014072,0.660206,0.673257,0.822202,0.779851
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.034232,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.494142,0.881428,0.467038,0.207062,0.363557,0.432848,0.840891,0.863852,0.182442,0.139867,...,0.507708,0.256781,0.172178,0.361305,0.799746,0.681716,0.179998,0.153395,0.354204,0.421773
75%,1.221919,1.947599,1.456434,0.943818,1.060823,1.047146,2.140746,1.927145,0.775049,0.781137,...,1.404729,0.975606,0.918453,1.074733,1.954997,1.540748,0.755484,0.740666,0.915391,1.132738
max,9.515568,10.547268,10.585348,4.040334,8.502610,7.389434,7.196245,4.491836,5.761020,5.497389,...,10.649902,4.965830,7.902304,9.622929,6.588682,4.495841,5.774849,5.346751,7.671851,4.220977


In [7]:
# Save the prepared asymmetric dataset
sustain_ready_df.to_csv(output_path, index=False)
print(f"Asymmetric dataset exported to: {output_path}")

Asymmetric dataset exported to: /Users/khoale/Downloads/Alzheimer_Code/adni_mri_sustain_prepared_asymmetric_as.csv
